# Ascent Path Formatting
Ascent supports an number of special keyword values that can be used in formatting path names to
generate unique file paths. 

[Path String Formatting](https://ascent.readthedocs.io/en/latest/Actions/Path_string_Formatting.html).

The formatting uses c++ generic ``sprintf()`` formatting as a backend so it will work exactly like
regular string formatting but follow a specific syntax for keyword support. To use Ascent's keyword
formatting, insert the desired keyword in curly braces ``{}`` in your file path string optionally
followed by a colon and the format specifier. 

Syntax: ``{keyword:format}``

If no format specifier is given, the default format for each keyword will be used.

These examples demonstrate using this keyword path formatting convention.

In [ ]:
# conduit + ascent imports
import conduit
import conduit.blueprint
import sys
import ascent

# cleanup any old results
!./cleanup.sh

### Create an example mesh to feed to Ascent

In [ ]:
# create example mesh using the conduit blueprint braid helper
mesh = conduit.Node()
conduit.blueprint.mesh.examples.braid("hexs",
                                      25,
                                      25,
                                      25,
                                      mesh)

## Path Formatting Example 1
### Using the ``cycle`` keyword

The cycle value represents the current simulation cycle. This is the default value used for
formatting meaning that when standard formatting convention is used in the path, such as ``%03d``,
the cycle value will be inserted at runtime. Additionally, in cases where no formatting is provided,
the cycle value will be appended to the end of the path name. 

``Default Format: 06d``

For this example, we will output the rendered scenes to three different paths to demonstrate
how the cycle value can be formatted in path names:
  - Default when no formatting:  ``out_fmt_cycle-no_fmt``
  - Standard formatting: ``out_fmt_cycle-standard_%04``
  - Ascent formatting:   ``out_fmt_cycle-ascent_fmt_{cycle:08d}``

In [ ]:
# These are the ascent actions that will be run on trigger
! cat "fmt_cycle_actions.yaml"

In [ ]:
# Use Ascent to bin an input mesh in a few ways
a = ascent.Ascent()

# open ascent
a.open()

# publish mesh to ascent
a.publish(mesh)

# setup actions
actions = conduit.Node()

# declare triggers 
add_triggers = actions.append()
add_triggers["action"] = "add_triggers"
triggers = add_triggers["triggers"] 

# add a simple trigger that fires for cycle valudes divisible by 200 (200, 400, ...)
triggers["t1/params/condition"] = "cycle() % 200 == 0"
triggers["t1/params/actions_file"] = "fmt_cycle_actions.yaml"

# view our full actions tree
print(actions.to_yaml())

# gyre time varying params
nsteps = 10
time = 0.0
delta_time = 0.5

for step in range(nsteps):    
    # update the example cycle
    cycle = 100 + step * 100
    mesh["state/cycle"] = cycle
    mesh["state/time"] = time
    print("time: {} cycle: {}".format(time,cycle))
    
    # publish mesh to ascent
    a.publish(mesh)
    
    # execute the actions
    a.execute(actions)
    
    # update time
    time = time + delta_time

# retrieve the info node that contains the trigger and query results
info = conduit.Node()
a.info(info)

# close ascent
a.close()

In [ ]:
# We expect our cycle trigger to render on cycles 200, 400, 600, 800, and 1000

# Recall that we rendered scenes to three different paths to demonstrate
# how the cycle value can be formatted in path names:
#     - Default when no formatting:  ``out_fmt_cycle-no_fmt``
#     - Standard formatting:         ``out_fmt_cycle-standard_%04``
#     - Ascent formatting:           ``out_fmt_cycle-ascent_fmt_{cycle:08d}``

! ls out_fmt_cycle*.png

In [ ]:
# cleanup any old results
!./cleanup.sh

## Path Formatting Example 2
### Using the ``time`` keyword

The time value represents the current simulation time as a floating point value.

``Default Format: g``

For this example, we will output the rendered scenes to three different paths to demonstrate different
formats that can be passed to the ascent string formatter:
  - Default Acent formatting:   ``out_fmt_time-default_{time}``
  - Float Ascent formatting:   ``out_fmt_time-float_{time:05.1f}``
  - Integer Ascent formatting:   ``out_fmt_time-int_{time:04d}``

In [ ]:
! cat "fmt_time_actions.yaml"

In [ ]:
# Use Ascent to bin an input mesh in a few ways
a = ascent.Ascent()

# open ascent
a.open()

# publish mesh to ascent
a.publish(mesh)

# setup actions
actions = conduit.Node()

# declare triggers 
add_triggers = actions.append()
add_triggers["action"] = "add_triggers"
triggers = add_triggers["triggers"] 

# add a simple trigger that fires for cycle valudes divisible by 300 (300, 600, ...)
triggers["t1/params/condition"] = "cycle() % 300 == 0"
triggers["t1/params/actions_file"] = "fmt_time_actions.yaml"

# view our full actions tree
print(actions.to_yaml())

# gyre time varying params
nsteps = 10
time = 0.0
delta_time = 0.5

for step in range(nsteps):    
    # update the example cycle
    cycle = 100 + step * 100
    mesh["state/cycle"] = cycle
    mesh["state/time"] = time
    print("time: {} cycle: {}".format(time,cycle))
    
    # publish mesh to ascent
    a.publish(mesh)
    
    # execute the actions
    a.execute(actions)
    
    # update time
    time = time + delta_time

# retrieve the info node that contains the trigger and query results
info = conduit.Node()
a.info(info)

# close ascent
a.close()

In [ ]:
# We expect our cycle trigger to render on cycles 300, 600, and 900
# These cycles coorespond with the times 1.0, 2.5, and 4.0

# Recall that we rendered scenes to three different paths to demonstrate different
#formats that can be passed to the ascent string formatter:
#     - Default Acent formatting:   ``out_fmt_time-default_{time}``
#     - Float Ascent formatting:    ``out_fmt_time-float_{time:05.1f}``
#     - Integer Ascent formatting:  ``out_fmt_time-int_{time:04d}``

! ls out_fmt_time*.png

In [ ]:
# cleanup any old results
!./cleanup.sh

## Path Formatting Example 3
### Using the ``family`` keyword

The family value is a unique value based on existing files in the output directory.

It is used to avoid overwriting existing files in the output directory with a
matching output file pattern, including those from previous runs of Ascent. It is set to one
greater than the maximum detected family value of the matching file pattern, or zero if no
matching files are found.

``Default Format: 06d``

For this example, we will only be outputting to one path. You can run the example multiple times
to see how the family value behavior persists across runs of ascent.
  - Output image prefix:   ``out_fmt_family_{family:03d}``

In [ ]:
# Use Ascent to bin an input mesh in a few ways
a = ascent.Ascent()

# open ascent
a.open()

# publish mesh to ascent
a.publish(mesh)

# setup actions
actions = conduit.Node()
add_act = actions.append()
add_act["action"] = "add_scenes"

# declare two scenes (s1 and s2) to render the dataset
scenes = add_act["scenes"]

scenes["s1/plots/p1/type"] = "pseudocolor"
scenes["s1/plots/p1/field"] = "braid"
scenes["s1/image_prefix"] = "out_fmt_family_{family:03d}"

# view our full actions tree
print(actions.to_yaml())

# Run ascent a couple times to demonstrate the family value behavior
for _ in range(5):    
    # execute the actions
    a.execute(actions)

# retrieve the info node that contains the query results
info = conduit.Node()
a.info(info)

# close ascent
a.close()

In [ ]:
# We expect to see multiple files each ending with a unique value.
# Note that the more times you run the above example the more files will be generated.

! ls out_fmt_family_*.png

In [ ]:
# cleanup any old results
!./cleanup.sh